#  C++ Primer : Writing Output into a File
# 1. Why
* Most measurements in this course end up as a file of numbers that something else --- a plotting script, a visualizer, your own eyes --- reads afterwards
* There are two ways to produce one, and you should know both

# 2. The shell way : redirection
* `printf` writes to _standard output_. The shell can point standard output at a file
```
./prog > a.dat
```
* Nothing in the program changes. This is the quickest way, and it is what the scheduling-visualization exercises in this course do
* The catch is that _everything_ printed goes into the file, including any message you meant for yourself. The remedy is `fprintf(stderr, ...)`, which writes to standard _error_ and is not redirected by `>`
```
fprintf(stderr, "n = %ld, starting\n", n);   /* appears on screen, not in a.dat */
```

# 3. The program way : `fopen`
```
FILE * fp = fopen("a.dat", "w");
if (!fp) { perror("a.dat"); return 1; }
fprintf(fp, "%ld %f\n", i, x);
fclose(fp);
```

* `fopen(name, mode)` opens a file and gives back a `FILE *`, the handle you pass to `fprintf`. Modes: `"w"` write (truncating whatever was there), `"a"` append, `"r"` read
* It returns a null pointer on failure --- no exception, nothing thrown. `if (!fp)` is the check, and `perror("a.dat")` prints the reason (`No such file or directory`, `Permission denied`, ...). Code that skips this check reports success while writing nothing
* `fprintf(fp, ...)` is `printf` with a destination. In fact `printf(...)` is `fprintf(stdout, ...)`
* `fclose(fp)` flushes the buffer and releases the handle. Output is buffered, so a program that is killed before `fclose` may leave a short or empty file --- a common and confusing way to lose a measurement
* Use this way when the program writes several files, or when it writes data and messages at once


In [ ]:
%%writefile out.cc
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

int main(int argc, char ** argv) {
  long n            = (argc > 1 ? atol(argv[1]) : 10);
  const char * file = (argc > 2 ? argv[2] : "a.dat");

  FILE * fp = fopen(file, "w");
  if (!fp) {
    perror(file);
    return 1;
  }
  fprintf(stderr, "writing %ld lines to %s\n", n, file);

  for (long i = 0; i < n; i++) {
    double x = i * 0.1;
    fprintf(fp, "%ld %f %f\n", i, x, sin(x));
  }
  fclose(fp);
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -Wall out.cc -o out

In [ ]:
%%bash
./out 5
echo ---- a.dat ----
cat a.dat

* The message went to the screen and the numbers went to the file, from the same program
* The file is an ordinary file in the notebook's directory, so anything can read it afterwards --- including Python in another cell of this notebook


In [ ]:
for line in open("a.dat"):
    i, x, y = line.split()
    print(i, float(y))

* <font color="green">Exercise: (a) run `./out 5 /no/such/dir/a.dat` and check both the message and the exit status (`echo $?`); then delete the `if (!fp)` check and run it again --- what does the program do now? (b) make the program write its numbers to standard output instead, and produce the same file with `>`; where does the `writing ... lines` message end up in each version? (c) write a second file, `b.dat`, holding only the lines with `sin(x) > 0`.</font>

* Start from this copy


In [ ]:
%%writefile out_ex.cc
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

int main(int argc, char ** argv) {
  long n            = (argc > 1 ? atol(argv[1]) : 10);
  const char * file = (argc > 2 ? argv[2] : "a.dat");

  FILE * fp = fopen(file, "w");
  if (!fp) {                     /* (a) try removing this check */
    perror(file);
    return 1;
  }
  /* (c) FILE * gp = fopen("b.dat", "w"); ... */

  fprintf(stderr, "writing %ld lines to %s\n", n, file);
  for (long i = 0; i < n; i++) {
    double x = i * 0.1;
    fprintf(fp, "%ld %f %f\n", i, x, sin(x));   /* (b) try stdout instead of fp */
    /* (c) if (sin(x) > 0) { ... } */
  }
  fclose(fp);
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -Wall out_ex.cc -o out_ex

In [ ]:
%%bash
./out_ex 5
echo "exit status = $?"